# Multi-Touch Attribution Models — Markov Removal-Effect & Shapley Value

**Why multi-touch, not last-click:** Diagnostics (`Diagnose.ipynb`) and EDA
(`EDA2.ipynb`) both show that ~79% of users touch 2+ channels before converting, and
that Google Search appears in 71.5% of converting journeys despite only being the
*last* touch in 48.4% of them. Last-click attribution would systematically undercount
every channel that does upper-funnel work. Two independent multi-touch models are
implemented here:

- **Markov chain removal-effect** — models the user journey as a Markov chain over
  channel states, then measures each channel's importance by how much the
  predicted conversion probability *drops* when that channel is removed from the
  chain entirely.
- **Shapley value** — a game-theoretic approach that treats each channel as a
  "player" and fairly distributes conversion credit based on its marginal
  contribution across every possible combination of channels.

Running both side-by-side (rather than picking one) lets the results be
cross-validated against each other — if they agree, that's stronger evidence of a
real signal; if they disagree, the disagreement itself is informative (see per-brand
results below).

**Input:** `touchpoints_clean_v3.csv`
**Output:** `attribution_overall_v2.csv`, `attribution_per_brand.csv`
(`per_brand_attribution.csv` used downstream in `CPA_calculation.ipynb`)

In [2]:
import pandas as pd
import numpy as np
import math
from itertools import combinations, chain

## Step 1 — Load data and extract brand IDs

`brand_id` is parsed out of `campaign_id` (pattern `CMP_<BRAND>_<CHANNEL>_<NUM>`) so
attribution can be computed both overall and broken out per brand, as required.

In [5]:
df = pd.read_csv('touchpoints_clean_v3.csv', parse_dates=['timestamp'])
df['channel']    = df['channel'].str.strip().str.title()
df['event_type'] = df['event_type'].str.strip()
df['brand_id']   = df['campaign_id'].str.extract(r'_(B\d+)_')
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)


In [6]:
channels = sorted(df['channel'].unique().tolist())
brands   = sorted(df['brand_id'].dropna().unique().tolist())
print(f"Channels: {channels}")
print(f"Brands:   {brands}\n")

Channels: ['Google Search', 'Influencer Blog', 'Instagram', 'Marketplace', 'Youtube']
Brands:   ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10']



## Step 2 — Build user journeys

Collapse each user's touchpoint rows into a single ordered list of channels touched
(`path`) plus a boolean `converted` flag. This journey representation is the shared
input for both the Markov and Shapley models below.

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# BUILD JOURNEYS (per user, channel sequence + conversion flag)
# ══════════════════════════════════════════════════════════════════════════
def build_journeys(data):
    def journey(g):
        return pd.Series({
            'path':      g['channel'].tolist(),
            'converted': (g['event_type'] == 'Purchase').any()
        })
    return data.groupby('user_id', group_keys=False).apply(journey)

## Step 3 — Markov chain removal-effect

**How it works:**
1. Build a transition matrix over states `Start → [channels] → Conversion / Null`,
   counting how often journeys move from one channel to the next (or to
   Conversion/Null at the end of the path).
2. Compute the baseline probability of reaching `Conversion` from `Start` by
   propagating the transition matrix forward.
3. For each channel, remove it from the chain (zero out its row and column,
   re-normalize) and recompute the conversion probability. The **drop** in
   conversion probability when a channel is removed is that channel's "removal
   effect" — its causal importance to the funnel.
4. Normalize removal effects across channels so they sum to 100%.

**Caveat baked into the code:** if a brand/segment has zero total removal effect
(e.g., too few distinct paths to compute a meaningful effect), the function falls
back to an equal split across channels rather than dividing by zero or returning
undefined credit.

In [13]:
# ══════════════════════════════════════════════════════════════════════════
# MARKOV CHAIN — REMOVAL EFFECT (run once on full data, then per brand)
# ══════════════════════════════════════════════════════════════════════════
def markov_attribution(journeys, channels):
    states = ['Start'] + channels + ['Conversion', 'Null']
    trans_counts = pd.DataFrame(0, index=states, columns=states, dtype=float)

    converting     = journeys[journeys['converted']]['path'].tolist()
    non_converting = journeys[~journeys['converted']]['path'].tolist()

    def count_transitions(paths, end_state):
        for path in paths:
            if not path:
                continue
            trans_counts.loc['Start', path[0]] += 1
            for i in range(len(path) - 1):
                trans_counts.loc[path[i], path[i+1]] += 1
            trans_counts.loc[path[-1], end_state] += 1

    count_transitions(converting, 'Conversion')
    count_transitions(non_converting, 'Null')

    row_sums = trans_counts.sum(axis=1)
    trans_matrix = trans_counts.div(row_sums.where(row_sums > 0), axis=0).fillna(0)

    def conversion_prob(tmat, max_steps=100):
        sv = np.zeros(len(states))
        sv[states.index('Start')] = 1.0
        T = tmat.values
        ci, ni = states.index('Conversion'), states.index('Null')
        total = 0.0
        for _ in range(max_steps):
            sv = sv @ T
            total += sv[ci]
            sv[ci] = 0
            sv[ni] = 0
            if sv.sum() < 1e-10:
                break
        return total

    baseline = conversion_prob(trans_matrix)
    removal = {}
    for ch in channels:
        red = trans_counts.copy()
        red.loc[ch, :] = 0
        red.loc[:, ch] = 0
        rs = red.sum(axis=1)
        rmat = red.div(rs.where(rs > 0), axis=0).fillna(0)
        removal[ch] = max(baseline - conversion_prob(rmat), 0)

    total = sum(removal.values())
    if total == 0:
        return {ch: 1/len(channels) for ch in channels}, baseline  # equal split fallback
    return {ch: v/total for ch, v in removal.items()}, baseline

## Step 4 — Shapley value attribution

**How it works:**
1. For every possible subset (coalition) of channels, compute that coalition's
   conversion rate: the share of journeys whose channels are *entirely contained*
   within the coalition that converted.
2. For each channel, compute its Shapley value — the weighted average of its
   marginal contribution (conversion-rate lift) across every coalition it could
   join, weighted by the standard Shapley combinatorial weights
   `s! (n-s-1)! / n!`.
3. Negative marginal contributions are clipped to 0 (a channel can't be assigned
   *negative* credit for being present), then values are normalized to sum to 100%.

**Cost of this approach:** Shapley value requires evaluating *every* subset of
channels (2⁵ = 32 coalitions here), which is exponential in the number of channels.
With only 5 channels this is cheap; it would not scale to a dataset with, say, 20+
channels without approximation (e.g., Monte Carlo sampling of permutations).

In [16]:
# ══════════════════════════════════════════════════════════════════════════
# SHAPLEY VALUE
# ══════════════════════════════════════════════════════════════════════════
def shapley_attribution(journeys, channels):
    n = len(channels)

    def coalition_rate(S, journeys):
        if not S:
            return 0.0
        subset = journeys[journeys['path'].apply(lambda p: bool(p) and set(p).issubset(S))]
        return subset['converted'].mean() if len(subset) else 0.0

    powerset = chain.from_iterable(combinations(channels, r) for r in range(1, n+1))
    coal_values = {frozenset(c): coalition_rate(frozenset(c), journeys) for c in powerset}

    shapley = {}
    for ch in channels:
        phi = 0.0
        others = [c for c in channels if c != ch]
        for r in range(len(others) + 1):
            for subset in combinations(others, r):
                S    = frozenset(subset)
                S_i  = S | {ch}
                s    = len(S)
                v_S, v_Si = coal_values.get(S, 0.0), coal_values.get(S_i, 0.0)
                weight = math.factorial(s) * math.factorial(n-s-1) / math.factorial(n)
                phi += weight * (v_Si - v_S)
        shapley[ch] = max(phi, 0)

    total = sum(shapley.values())
    return {ch: (v/total if total > 0 else 1/n) for ch, v in shapley.items()}


## Step 5 — Run: overall attribution (all brands combined)

Compares last-click, Markov, and Shapley side by side. The contrast is stark:

- **Last-click** gives Google Search 48.4% of credit (because it's disproportionately
  the final touch before purchase).
- **Markov removal-effect** gives Google Search 100% of credit overall — removing
  Search collapses the conversion probability entirely, which makes sense if Search
  sits on nearly every path as a near-universal gateway step in the aggregate data.
- **Shapley value** is more moderate (50.5% Search, with Influencer Blog, Instagram,
  and Marketplace picking up meaningful shares) — because it credits channels for
  their contribution within *every* coalition, not just full-chain removal.

The Markov model's "100% to one channel" result at the aggregate level is a real
limitation worth flagging: when one channel is structurally present in nearly every
path, full-removal can produce an all-or-nothing answer that's directionally correct
but too extreme to act on directly. This is part of the motivation for also running
Shapley, and for breaking results out per brand next, where the picture changes
substantially.

In [19]:
# ══════════════════════════════════════════════════════════════════════════
# RUN: OVERALL (ALL BRANDS COMBINED)
# ══════════════════════════════════════════════════════════════════════════
print("="*70)
print("OVERALL ATTRIBUTION (all brands combined)")
print("="*70)

all_journeys = build_journeys(df)
print(f"Total journeys: {len(all_journeys)} | Converting: {all_journeys['converted'].sum()}")

markov_overall, baseline_conv = markov_attribution(all_journeys, channels)
print(f"\nBaseline conversion probability: {baseline_conv:.6f}")
print("\nMarkov Attribution (%):")
for ch, v in sorted(markov_overall.items(), key=lambda x: -x[1]):
    print(f"  {ch:<20} {v*100:.2f}%")

shapley_overall = shapley_attribution(all_journeys, channels)
print("\nShapley Attribution (%):")
for ch, v in sorted(shapley_overall.items(), key=lambda x: -x[1]):
    print(f"  {ch:<20} {v*100:.2f}%")

# Last-click for comparison
last_click_raw = df[df['event_type']=='Purchase'].groupby('channel').size()
last_click_pct = (last_click_raw / last_click_raw.sum())

overall_comparison = pd.DataFrame({
    'channel':         channels,
    'last_click_pct':  [round(last_click_pct.get(ch,0)*100, 2) for ch in channels],
    'markov_pct':      [round(markov_overall[ch]*100, 2) for ch in channels],
    'shapley_pct':     [round(shapley_overall[ch]*100, 2) for ch in channels],
}).sort_values('shapley_pct', ascending=False)

print("\n" + overall_comparison.to_string(index=False))
overall_comparison.to_csv('attribution_overall_v2.csv', index=False)

OVERALL ATTRIBUTION (all brands combined)


C:\Users\mohit\AppData\Local\Temp\ipykernel_3764\1519537219.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return data.groupby('user_id', group_keys=False).apply(journey)


Total journeys: 98048 | Converting: 5498

Baseline conversion probability: 0.056075

Markov Attribution (%):
  Google Search        100.00%
  Influencer Blog      0.00%
  Instagram            0.00%
  Marketplace          0.00%
  Youtube              0.00%

Shapley Attribution (%):
  Google Search        50.48%
  Influencer Blog      19.95%
  Instagram            13.74%
  Marketplace          11.73%
  Youtube              4.11%

        channel  last_click_pct  markov_pct  shapley_pct
  Google Search           48.40       100.0        50.48
Influencer Blog            9.11         0.0        19.95
      Instagram           15.88         0.0        13.74
    Marketplace           18.62         0.0        11.73
        Youtube            7.98         0.0         4.11


## Step 6 — Run: per-brand attribution (required by project brief)

Re-running both models separately for each of the 10 brands (skipping any brand with
fewer than 5 conversions, to avoid unstable estimates on too little data) reveals
that the aggregate "Search wins everything" story does **not** hold uniformly:

- Brand B01 is dominated by Instagram (Markov 100%, Shapley 77.7%), not Search.
- Brand B09 is dominated entirely by Marketplace.
- Brands B03–B06, B08, B10 show genuinely mixed, multi-channel credit under Shapley,
  even where Markov assigns most of the removal-effect to one or two channels.

**Markov vs. Shapley disagreement is itself a useful diagnostic:** where the two
models agree closely (e.g., B01, B02, B09), that's strong, well-corroborated evidence
about which channel matters. Where they diverge sharply (e.g., B07, where Markov
gives Search 94.9% but Shapley gives it 100% with literally 0% to every other
channel — versus B05, where Shapley spreads credit far more evenly than Markov),
that's a signal to treat the result with more caution, since the two methods are
making different structural assumptions about how credit should flow.

This per-brand `shapley_pct` output (saved as `per_brand_attribution.csv`) is the
direct input to the CPA calculation in `CPA_calculation.ipynb`, where it's combined
with actual spend to compute cost-per-attributed-conversion and flag defund /
frequency-cap candidates.

In [21]:
# ══════════════════════════════════════════════════════════════════════════
# RUN: PER BRAND (required by project brief — all 10 brands)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("PER-BRAND ATTRIBUTION (10 brands)")
print("="*70)

brand_results = []
for brand in brands:
    brand_df = df[df['brand_id'] == brand]
    brand_journeys = build_journeys(brand_df)

    if brand_journeys['converted'].sum() < 5:
        print(f"{brand}: too few conversions, skipping")
        continue

    m_attr, _ = markov_attribution(brand_journeys, channels)
    s_attr    = shapley_attribution(brand_journeys, channels)

    for ch in channels:
        brand_results.append({
            'brand_id':   brand,
            'channel':    ch,
            'markov_pct': round(m_attr[ch]*100, 2),
            'shapley_pct':round(s_attr[ch]*100, 2),
            'n_journeys': len(brand_journeys),
            'n_converted':int(brand_journeys['converted'].sum())
        })

brand_df_results = pd.DataFrame(brand_results)
print(brand_df_results.to_string(index=False))
brand_df_results.to_csv('attribution_per_brand.csv', index=False)

print("\n✓ Saved: attribution_overall_v2.csv")
print("✓ Saved: attribution_per_brand.csv")



PER-BRAND ATTRIBUTION (10 brands)


C:\Users\mohit\AppData\Local\Temp\ipykernel_3764\1519537219.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return data.groupby('user_id', group_keys=False).apply(journey)
C:\Users\mohit\AppData\Local\Temp\ipykernel_3764\1519537219.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return data.groupby('user_id', group_keys=False).apply(journey)
C:\Users\mohit\AppData\Local\Temp\ipykernel_3764\1519537

brand_id         channel  markov_pct  shapley_pct  n_journeys  n_converted
     B01   Google Search        0.00         2.31        9796          698
     B01 Influencer Blog        0.00        11.66        9796          698
     B01       Instagram      100.00        77.67        9796          698
     B01     Marketplace        0.00         1.27        9796          698
     B01         Youtube        0.00         7.10        9796          698
     B02   Google Search      100.00        46.05        9810         1277
     B02 Influencer Blog        0.00        44.02        9810         1277
     B02       Instagram        0.00         4.12        9810         1277
     B02     Marketplace        0.00         1.14        9810         1277
     B02         Youtube        0.00         4.66        9810         1277
     B03   Google Search       33.84        23.24        9820          303
     B03 Influencer Blog       66.16        29.86        9820          303
     B03       Instagram 